# LeetCode 1042 — Flower Planting With No Adjacent bằng Python

Link đề: https://leetcode.com/problems/flower-planting-with-no-adjacent/ · Độ khó: **Medium** · Chủ đề: Graph, DFS, BFS, Graph Coloring, Greedy

Skeleton LeetCode:

```python
class Solution:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đề cho **nhiều đáp án đúng** ("Return **any** such a choice"). Vì vậy notebook này **không** so sánh đáp án với một mảng cố định mà kiểm tra bằng **hàm hợp lệ** `is_valid(n, paths, ans)`: độ dài đúng `n`, mỗi giá trị thuộc `{1, 2, 3, 4}`, hai vườn có đường nối trực tiếp thì khác loại hoa. Điểm cốt lõi: mỗi vườn có **tối đa 3** hàng xóm mà có **4** loại hoa nên luôn còn ít nhất một loại hoa trống ⇒ **tô tham lam một lượt** là đủ, không cần quay lui.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` khu vườn đánh số `1..n`, danh sách đường đi hai chiều `paths`, mỗi vườn trồng một trong `4` loại hoa sao cho hai vườn nối nhau có loại hoa khác nhau; mỗi vườn có **tối đa `3`** đường.
2. Nắm công cụ: danh sách kề, tô màu đồ thị (graph coloring), tập "màu đã dùng", mặt nạ bit.
3. Đi từ **vét cạn `4^n`** → **quay lui** → **tô tham lam theo chỉ số** → **tô theo thứ tự bất kỳ (BFS)** → **mặt nạ bit**.
4. Hiểu mấu chốt: bậc `≤ 3` và có `4` màu ⇒ tại thời điểm tô một đỉnh, số hàng xóm đã tô `≤ 3 < 4` nên luôn còn màu trống. Tham lam **không bao giờ kẹt**, nên không cần quay lui.
5. Biết cách **test bài có nhiều đáp án**: viết hàm kiểm tra tính hợp lệ (validity), không so sánh bằng mảng mẫu.
6. Tránh lỗi kinh điển: chỉ thêm cạnh một chiều, chỉ dùng `3` màu, lệch chỉ số `1`-based/`0`-based, so sánh đáp án với mảng mẫu của đề, lấy màu theo `i % 4` mà không nhìn hàng xóm.

## 2. Đề bài & ví dụ

Có `n` khu vườn đánh số từ `1` đến `n` và mảng `paths`, trong đó `paths[i] = [x_i, y_i]` là một con đường **hai chiều** nối vườn `x_i` với vườn `y_i`. Ở mỗi vườn, bạn muốn trồng đúng một trong `4` loại hoa.

Mọi khu vườn có **nhiều nhất `3`** con đường đi vào hoặc đi ra.

Hãy chọn loại hoa cho từng vườn sao cho **hai vườn nối trực tiếp bằng một con đường có loại hoa khác nhau**. Trả về **một** phương án bất kỳ dưới dạng mảng `answer`, trong đó `answer[i]` là loại hoa (`1`, `2`, `3` hoặc `4`) trồng ở vườn thứ `i + 1`. Đề đảm bảo luôn tồn tại đáp án.

**Ví dụ 1:** `n = 3`, `paths = [[1,2],[2,3],[3,1]]` → `[1,2,3]` (cũng đúng: `[1,2,4]`, `[1,4,2]`, `[3,2,1]`).

**Ví dụ 2:** `n = 4`, `paths = [[1,2],[3,4]]` → `[1,2,1,2]`.

**Ví dụ 3:** `n = 4`, `paths = [[1,2],[2,3],[3,4],[4,1],[1,3],[2,4]]` → `[1,2,3,4]`.

**Ràng buộc:** `1 <= n <= 10^4`; `0 <= paths.length <= 2 * 10^4`; `1 <= x_i, y_i <= n`; `x_i != y_i`; mỗi vườn có tối đa `3` đường.

In [1]:
import itertools
import random
import time
from collections import defaultdict, deque
from typing import List

examples = [
    (3, [[1, 2], [2, 3], [3, 1]]),
    (4, [[1, 2], [3, 4]]),
    (4, [[1, 2], [2, 3], [3, 4], [4, 1], [1, 3], [2, 4]]),
]
sample_answers = [[1, 2, 3], [1, 2, 1, 2], [1, 2, 3, 4]]
for (n, paths), ans in zip(examples, sample_answers):
    print(f"n = {n}, paths = {paths} -> ví dụ đáp án {ans}")

n = 3, paths = [[1, 2], [2, 3], [3, 1]] -> ví dụ đáp án [1, 2, 3]
n = 4, paths = [[1, 2], [3, 4]] -> ví dụ đáp án [1, 2, 1, 2]
n = 4, paths = [[1, 2], [2, 3], [3, 4], [4, 1], [1, 3], [2, 4]] -> ví dụ đáp án [1, 2, 3, 4]


## 3. Kiến thức nền

### 3.1 Đồ thị và danh sách kề

Mỗi khu vườn là một **đỉnh**, mỗi con đường là một **cạnh vô hướng**. Danh sách kề `g[x]` chứa các đỉnh kề `x`. Với đường `[x, y]` ta phải thêm **cả hai chiều**: `g[x].append(y)` và `g[y].append(x)`. Ta chuyển sang chỉ số `0`-based (`x - 1`) để dùng thẳng làm chỉ số mảng.

In [2]:
def build_graph(n, paths):
    g = [[] for _ in range(n)]
    for x, y in paths:
        g[x - 1].append(y - 1)
        g[y - 1].append(x - 1)
    return g


g = build_graph(*examples[2])
print(g, "bậc:", [len(a) for a in g])

[[1, 3, 2], [0, 2, 3], [1, 3, 0], [2, 0, 1]] bậc: [3, 3, 3, 3]


### 3.2 Hàm kiểm tra tính hợp lệ (validity function)

Vì đáp án không duy nhất nên **không** viết `assert ans == [1,2,3]`. Thay vào đó viết hàm `is_valid` kiểm tra đúng định nghĩa của đề:

1. `ans` là danh sách độ dài `n`;
2. mỗi phần tử là số nguyên trong `{1, 2, 3, 4}`;
3. với mọi đường `[x, y]` thì `ans[x-1] != ans[y-1]`.

Hàm này là **oracle** của notebook: mọi cài đặt phải cho đáp án mà `is_valid` chấp nhận.

In [3]:
def is_valid(n, paths, ans):
    if not isinstance(ans, list) or len(ans) != n:
        return False
    if any(type(c) is not int or c < 1 or c > 4 for c in ans):
        return False
    return all(ans[x - 1] != ans[y - 1] for x, y in paths)


assert is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2, 3])
assert is_valid(3, [[1, 2], [2, 3], [3, 1]], [3, 2, 1]) and is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2, 4])
assert not is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2, 2])      # 2 và 3 cùng loại
assert not is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2])         # thiếu phần tử
assert not is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2, 5])      # loại hoa ngoài 1..4
assert not is_valid(3, [[1, 2], [2, 3], [3, 1]], [0, 2, 3])      # 0 nghĩa là chưa trồng
print("is_valid chấp nhận các đáp án đúng khác nhau và loại bỏ các đáp án sai")

is_valid chấp nhận các đáp án đúng khác nhau và loại bỏ các đáp án sai


### 3.3 Số hàng xóm và số màu

Gọi `Δ` là bậc lớn nhất. Một đỉnh có tối đa `Δ` hàng xóm nên tối đa `Δ` màu bị "cấm"; nếu có `Δ + 1` màu thì luôn còn một màu trống. Ở bài này `Δ ≤ 3` và có `4 = Δ + 1` màu — đúng ngưỡng vừa đủ. Đây là **định lý tô màu tham lam**: mọi đồ thị bậc tối đa `Δ` tô được bằng `Δ + 1` màu.

## 4. Ý tưởng nền tảng / trực giác

Hình dung ta đi qua từng khu vườn, theo thứ tự nào đó, và mỗi lần trồng hoa:

- Nhìn các hàng xóm **đã được trồng**, ghi lại loại hoa của họ.
- Chọn một loại hoa trong `1..4` **không nằm** trong tập đó.

Vì mỗi vườn chỉ có tối đa `3` hàng xóm, tập loại hoa bị cấm có tối đa `3` phần tử trong `4` loại ⇒ **luôn còn ít nhất một loại**. Do đó không bao giờ bị kẹt và không cần quay lui. Đề còn nói thẳng "đảm bảo có đáp án" — bậc `≤ 3` chính là lý do.

## 5. Brute force — thử mọi cách gán `4^n`

**Ý tưởng:** liệt kê mọi bộ `(c_1, …, c_n) ∈ {1,2,3,4}^n` theo thứ tự từ điển, trả về bộ **hợp lệ đầu tiên**. Đây là cách đúng theo định nghĩa, chỉ chạy nổi khi `n` rất nhỏ (`n ≤ 8`). Vì `itertools.product` đi theo thứ tự từ điển nên bộ tìm được chính là **đáp án nhỏ nhất theo từ điển** — ta sẽ dùng nó ở mục 10 và 15.

In [4]:
def garden_brute(n, paths):
    for ans in itertools.product((1, 2, 3, 4), repeat=n):
        if all(ans[x - 1] != ans[y - 1] for x, y in paths):
            return list(ans)
    return []                                              # không tồn tại (không xảy ra với đầu vào hợp lệ)


for (n, paths), _ in zip(examples, sample_answers):
    r = garden_brute(n, paths)
    assert is_valid(n, paths, r)
    print(f"n = {n}: brute -> {r}")

n = 3: brute -> [1, 2, 3]
n = 4: brute -> [1, 2, 1, 2]
n = 4: brute -> [1, 2, 3, 4]


In [5]:
# Code nộp LeetCode cho Brute force (chỉ đúng, KHÔNG qua được test lớn: 4^n)
class SolutionBrute:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        for ans in itertools.product((1, 2, 3, 4), repeat=n):
            if all(ans[x - 1] != ans[y - 1] for x, y in paths):
                return list(ans)
        return []

**Độ phức tạp:** `O(4^n · m)` thời gian (`m = len(paths)`), `O(n)` bộ nhớ.

**Nút thắt:** ta thử toàn bộ không gian gán trong khi mỗi đỉnh **luôn** có sẵn một màu hợp lệ nếu quyết định theo hàng xóm đã tô.

## 6. Bước đột phá

Gán màu **lần lượt từng đỉnh**: khi tới đỉnh `x`, chỉ cần tránh màu của các hàng xóm **đã** được tô. Vì có `≤ 3` hàng xóm và `4` màu nên màu trống luôn tồn tại; quyết định này không bao giờ làm đỉnh sau bị kẹt (đỉnh sau cũng có `≤ 3` hàng xóm). Vậy một vòng lặp `O(n + m)` là đủ.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** quay lui (backtracking), **Cách 3** tô tham lam theo chỉ số (đáp án chính), **Cách 4** tô tham lam theo thứ tự BFS (thứ tự bất kỳ), **Cách 5** tham lam bằng mặt nạ bit.

### Cách 2 — Quay lui (backtracking)

**Giải thích:** tô các đỉnh `0, 1, 2, …` theo thứ tự; ở đỉnh `i` thử màu `1..4`, chỉ chấp nhận màu khác mọi hàng xóm **có chỉ số nhỏ hơn** (đã tô). Nếu hết màu thì lùi về đỉnh trước và thử màu kế tiếp. Cài đặt bằng vòng lặp (không đệ quy) để không lo tràn stack. Đây là thuật toán tổng quát cho bài tô màu đồ thị; ở bài này nó **không bao giờ phải lùi** (ta đếm số lần lùi ở mục 10).

In [6]:
def garden_backtrack(n, paths, count_back=False):
    g = build_graph(n, paths)
    ans = [0] * n
    i, backs = 0, 0
    while 0 <= i < n:
        ans[i] += 1
        while ans[i] <= 4 and any(ans[y] == ans[i] for y in g[i] if y < i):
            ans[i] += 1                                    # màu này trùng một hàng xóm đã tô: thử màu kế
        if ans[i] > 4:                                     # hết màu: lùi về đỉnh trước
            ans[i] = 0
            i -= 1
            backs += 1
        else:
            i += 1
    res = ans if i == n else []
    return (res, backs) if count_back else res


for (n, paths), _ in zip(examples, sample_answers):
    r = garden_backtrack(n, paths)
    assert is_valid(n, paths, r)
    print(f"n = {n}: backtrack -> {r}")

n = 3: backtrack -> [1, 2, 3]
n = 4: backtrack -> [1, 2, 1, 2]
n = 4: backtrack -> [1, 2, 3, 4]


In [7]:
# Code nộp LeetCode cho Cách 2 (quay lui lặp)
class SolutionBacktrack:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = [[] for _ in range(n)]
        for x, y in paths:
            g[x - 1].append(y - 1)
            g[y - 1].append(x - 1)
        ans = [0] * n
        i = 0
        while 0 <= i < n:
            ans[i] += 1
            while ans[i] <= 4 and any(ans[y] == ans[i] for y in g[i] if y < i):
                ans[i] += 1
            if ans[i] > 4:
                ans[i] = 0
                i -= 1
            else:
                i += 1
        return ans if i == n else []

**Độ phức tạp:** trường hợp xấu tổng quát `O(4^n)`; ở bài này không lùi nên `O(n + m)` thời gian, `O(n + m)` bộ nhớ.

### Cách 3 — Tô tham lam theo chỉ số (đáp án chính)

**Ý tưởng:** dựng danh sách kề; với `x = 0..n-1` lấy tập `used` = màu của các hàng xóm (hàng xóm chưa tô có màu `0` nên không ảnh hưởng), rồi gán màu **nhỏ nhất trong `1..4`** không thuộc `used`.

In [8]:
def garden_greedy(n, paths):
    g = build_graph(n, paths)
    ans = [0] * n
    for x in range(n):
        used = {ans[y] for y in g[x]}                      # màu đã bị hàng xóm chiếm (0 = chưa tô, bỏ qua)
        for c in range(1, 5):
            if c not in used:
                ans[x] = c
                break
    return ans


for (n, paths), _ in zip(examples, sample_answers):
    r = garden_greedy(n, paths)
    assert is_valid(n, paths, r)
    print(f"n = {n}: greedy -> {r}")

n = 3: greedy -> [1, 2, 3]
n = 4: greedy -> [1, 2, 1, 2]
n = 4: greedy -> [1, 2, 3, 4]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionGreedy:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = defaultdict(list)
        for x, y in paths:
            x, y = x - 1, y - 1
            g[x].append(y)
            g[y].append(x)
        ans = [0] * n
        for x in range(n):
            used = {ans[y] for y in g[x]}
            for c in range(1, 5):
                if c not in used:
                    ans[x] = c
                    break
        return ans

**Độ phức tạp:** `O(n + m)` thời gian (mỗi đỉnh duyệt hàng xóm `≤ 3` và thử `≤ 4` màu), `O(n + m)` bộ nhớ.

### Cách 4 — Tham lam theo thứ tự BFS (thứ tự bất kỳ)

**Ý tưởng:** chứng minh ở mục 10 không phụ thuộc thứ tự thăm đỉnh. Ta duyệt BFS từng thành phần liên thông để có một thứ tự đỉnh, rồi tô tham lam theo thứ tự đó. Hàm `color_in_order` nhận **thứ tự tuỳ ý**, nên ta sẽ test với hoán vị ngẫu nhiên ở mục 12.

In [10]:
def color_in_order(g, order):
    """Tô tham lam theo thứ tự `order` (một hoán vị bất kỳ của 0..n-1)."""
    ans = [0] * len(g)
    for x in order:
        used = {ans[y] for y in g[x]}
        ans[x] = next(c for c in (1, 2, 3, 4) if c not in used)
    return ans


def bfs_order(g):
    seen = [False] * len(g)
    order = []
    for s in range(len(g)):
        if seen[s]:
            continue
        seen[s] = True
        dq = deque([s])
        while dq:
            u = dq.popleft()
            order.append(u)
            for v in g[u]:
                if not seen[v]:
                    seen[v] = True
                    dq.append(v)
    return order


def garden_bfs(n, paths):
    g = build_graph(n, paths)
    return color_in_order(g, bfs_order(g))


for (n, paths), _ in zip(examples, sample_answers):
    r = garden_bfs(n, paths)
    assert is_valid(n, paths, r)
    print(f"n = {n}: bfs-order -> {r}")

n = 3: bfs-order -> [1, 2, 3]
n = 4: bfs-order -> [1, 2, 1, 2]
n = 4: bfs-order -> [1, 2, 4, 3]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionBFS:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = [[] for _ in range(n)]
        for x, y in paths:
            g[x - 1].append(y - 1)
            g[y - 1].append(x - 1)
        ans = [0] * n
        for s in range(n):
            if ans[s]:
                continue
            q = deque([s])
            seen = {s}
            while q:
                u = q.popleft()
                used = {ans[v] for v in g[u]}
                ans[u] = next(c for c in (1, 2, 3, 4) if c not in used)
                for v in g[u]:
                    if v not in seen and not ans[v]:
                        seen.add(v)
                        q.append(v)
        return ans

**Lưu ý code nộp:** trong `SolutionBFS`, `seen` chỉ dùng cục bộ trong từng thành phần liên thông; đỉnh được tô đúng một lần (lúc `pop`), tại thời điểm đó mọi hàng xóm đã tô đều được tránh nên vẫn hợp lệ.

**Độ phức tạp:** `O(n + m)` thời gian, `O(n + m)` bộ nhớ.

### Cách 5 — Tham lam với mặt nạ bit

**Ý tưởng:** thay tập `used` bằng một số nguyên `mask`, bit `c` bật nếu màu `c` đã bị dùng. Màu chưa tô là `0` (bit `0`, ta bỏ qua bằng cách chỉ xét các bit `1..4`). Màu trống nhỏ nhất là bit thấp nhất của `~mask & 0b11110`, lấy bằng `free & -free`.

In [12]:
def garden_bitmask(n, paths):
    g = build_graph(n, paths)
    ans = [0] * n
    for x in range(n):
        mask = 0
        for y in g[x]:
            mask |= 1 << ans[y]                            # bật bit của màu hàng xóm (bit 0 = chưa tô)
        free = ~mask & 0b11110                             # chỉ xét màu 1..4
        ans[x] = (free & -free).bit_length() - 1           # chỉ số bit thấp nhất = màu nhỏ nhất còn trống
    return ans


for (n, paths), _ in zip(examples, sample_answers):
    r = garden_bitmask(n, paths)
    assert is_valid(n, paths, r)
    print(f"n = {n}: bitmask -> {r}")

n = 3: bitmask -> [1, 2, 3]
n = 4: bitmask -> [1, 2, 1, 2]
n = 4: bitmask -> [1, 2, 3, 4]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionBitmask:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = [[] for _ in range(n)]
        for x, y in paths:
            g[x - 1].append(y - 1)
            g[y - 1].append(x - 1)
        ans = [0] * n
        for x in range(n):
            mask = 0
            for y in g[x]:
                mask |= 1 << ans[y]
            free = ~mask & 0b11110
            ans[x] = (free & -free).bit_length() - 1
        return ans

**Độ phức tạp:** `O(n + m)` thời gian, `O(n + m)` bộ nhớ; hằng số nhỏ hơn dùng `set`.

## 8. TRACE từng bước

Trace **Cách 3** với ví dụ 3: `n = 4`, `paths = [[1,2],[2,3],[3,4],[4,1],[1,3],[2,4]]` (đồ thị đầy đủ `K4`, mỗi đỉnh bậc `3`).

| vườn `x` | hàng xóm (0-based) | màu hàng xóm lúc đó | `used` | chọn |
|---|---|---|---|---|
| 0 | 1, 3, 2 | 0, 0, 0 | {0} | 1 |
| 1 | 0, 2, 3 | 1, 0, 0 | {0, 1} | 2 |
| 2 | 1, 3, 0 | 2, 0, 1 | {0, 1, 2} | 3 |
| 3 | 2, 0, 1 | 3, 1, 2 | {1, 2, 3} | 4 |

`K4` cần đúng `4` màu: vườn cuối có `3` hàng xóm đã tô, chỉ còn màu `4`. Hàm trace in từng bước:

In [14]:
def trace_greedy(n, paths):
    g = build_graph(n, paths)
    ans = [0] * n
    print(f"n = {n}, paths = {paths}")
    for x in range(n):
        used = {ans[y] for y in g[x]}
        c = next(c for c in range(1, 5) if c not in used)
        nb_info = ", ".join(f"{y + 1}:{ans[y]}" for y in g[x]) or "(không có)"
        print(f"  vườn {x + 1}: hàng xóm (vườn:màu) {nb_info}; màu bị cấm {sorted(used - {0})} -> chọn {c}")
        ans[x] = c
    print("  kết quả:", ans, "| hợp lệ:", is_valid(n, paths, ans))
    return ans


assert trace_greedy(*examples[2]) == [1, 2, 3, 4]
assert trace_greedy(*examples[1]) == [1, 2, 1, 2]

n = 4, paths = [[1, 2], [2, 3], [3, 4], [4, 1], [1, 3], [2, 4]]
  vườn 1: hàng xóm (vườn:màu) 2:0, 4:0, 3:0; màu bị cấm [] -> chọn 1
  vườn 2: hàng xóm (vườn:màu) 1:1, 3:0, 4:0; màu bị cấm [1] -> chọn 2
  vườn 3: hàng xóm (vườn:màu) 2:2, 4:0, 1:1; màu bị cấm [1, 2] -> chọn 3
  vườn 4: hàng xóm (vườn:màu) 3:3, 1:1, 2:2; màu bị cấm [1, 2, 3] -> chọn 4
  kết quả: [1, 2, 3, 4] | hợp lệ: True
n = 4, paths = [[1, 2], [3, 4]]
  vườn 1: hàng xóm (vườn:màu) 2:0; màu bị cấm [] -> chọn 1
  vườn 2: hàng xóm (vườn:màu) 1:1; màu bị cấm [1] -> chọn 2
  vườn 3: hàng xóm (vườn:màu) 4:0; màu bị cấm [] -> chọn 1
  vườn 4: hàng xóm (vườn:màu) 3:1; màu bị cấm [1] -> chọn 2
  kết quả: [1, 2, 1, 2] | hợp lệ: True


**Trace chu trình độ dài 5** (`paths = [[1,2],[2,3],[3,4],[4,5],[5,1]]`): các đỉnh `1..4` lần lượt được `1, 2, 1, 2`; đỉnh `5` có hàng xóm `4` (màu `2`) và `1` (màu `1`) nên bị cấm `{1, 2}` và nhận màu `3`. Một đồ thị `2`-màu không đủ cho chu trình lẻ, nhưng `3` màu là đủ.

In [15]:
assert trace_greedy(5, [[1, 2], [2, 3], [3, 4], [4, 5], [5, 1]]) == [1, 2, 1, 2, 3]

n = 5, paths = [[1, 2], [2, 3], [3, 4], [4, 5], [5, 1]]
  vườn 1: hàng xóm (vườn:màu) 2:0, 5:0; màu bị cấm [] -> chọn 1
  vườn 2: hàng xóm (vườn:màu) 1:1, 3:0; màu bị cấm [1] -> chọn 2
  vườn 3: hàng xóm (vườn:màu) 2:2, 4:0; màu bị cấm [2] -> chọn 1
  vườn 4: hàng xóm (vườn:màu) 3:1, 5:0; màu bị cấm [1] -> chọn 2
  vườn 5: hàng xóm (vườn:màu) 4:2, 1:1; màu bị cấm [1, 2] -> chọn 3
  kết quả: [1, 2, 1, 2, 3] | hợp lệ: True


## 9. Code nộp LeetCode

Chọn **Cách 3** (tô tham lam theo chỉ số). Giữ nguyên `class Solution`, `gardenNoAdj(self, n, paths)`:

In [16]:
class Solution:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = defaultdict(list)                       # danh sách kề, chỉ số 0-based
        for x, y in paths:
            x, y = x - 1, y - 1
            g[x].append(y)                          # đường hai chiều: thêm CẢ HAI hướng
            g[y].append(x)
        ans = [0] * n                               # 0 = chưa trồng
        for x in range(n):
            used = {ans[y] for y in g[x]}           # loại hoa các hàng xóm (đã trồng) đang dùng
            for c in range(1, 5):                   # bậc <= 3 nên chắc chắn còn ít nhất một loại trong 1..4
                if c not in used:
                    ans[x] = c
                    break
        return ans


for n, paths in examples:
    r = Solution().gardenNoAdj(n, paths)
    assert is_valid(n, paths, r)
    print(n, paths, "->", r)

3 [[1, 2], [2, 3], [3, 1]] -> [1, 2, 3]
4 [[1, 2], [3, 4]] -> [1, 2, 1, 2]
4 [[1, 2], [2, 3], [3, 4], [4, 1], [1, 3], [2, 4]] -> [1, 2, 3, 4]


## 10. Vì sao đúng

**(a) Bất biến.** Xét thời điểm tô đỉnh `x` (theo thứ tự bất kỳ). Gọi `N(x)` là tập hàng xóm, `|N(x)| ≤ 3`. Màu bị cấm là `{ans[y] : y ∈ N(x), ans[y] ≠ 0}` có không quá `|N(x)| ≤ 3` phần tử, trong khi có `4` màu. Do đó tồn tại màu `c ∈ {1,2,3,4}` không bị cấm, thuật toán luôn gán được; vòng `for c in range(1, 5)` không bao giờ kết thúc mà không gán.

**(b) Tính hợp lệ.** Giả sử hai đỉnh kề `x, y` với `x` được tô trước `y`. Khi tô `y`, `x ∈ N(y)` đã có màu khác `0` nên màu của `x` nằm trong `used` của `y`; `y` được chọn màu **không thuộc** `used` ⇒ màu `y ≠` màu `x`. Vì mọi cạnh có một đầu mút tô trước, mọi cạnh đều thoả. Đây chính là điều `is_valid` kiểm tra. (Chiều ngược `x` kiểm `y` không cần vì `y` chưa tô, màu `0` bị bỏ qua; nếu quên thêm cạnh hai chiều thì `y` sẽ không thấy `x` — lỗi 1 ở mục 13.)

**(c) Độc lập với thứ tự.** Chứng minh (a), (b) không dùng thứ tự nào cả ⇒ mọi hoán vị đỉnh đều cho đáp án hợp lệ: Cách 4 đúng.

**(d) Quay lui không lùi.** Ở Cách 2, tại đỉnh `i` các hàng xóm đã tô là tập con của `N(i)` nên có màu trống; vòng `while` luôn dừng với `ans[i] ≤ 4` ⇒ không bao giờ lùi, và Cách 2 cho **đúng cùng đáp án** với Cách 3 (cùng chọn màu nhỏ nhất không bị cấm).

**(e) Đáp án nhỏ nhất theo từ điển.** Mọi tiền tố tô hợp lệ đều **mở rộng được** (theo (a)). Tham lam chọn tại vị trí `i` màu **nhỏ nhất** không xung đột với các vị trí trước; bất kỳ đáp án hợp lệ nào có cùng tiền tố đó buộc `ans[i] ≥` màu này; và màu này vẫn mở rộng được. Quy nạp ⇒ đáp án tham lam theo chỉ số = **đáp án nhỏ nhất theo từ điển** = bộ hợp lệ đầu tiên của vét cạn `product` (mục 5). Ta kiểm bằng máy ở mục 12.

**(f) Tại sao `4` màu là vừa đủ.** Đồ thị đầy đủ `K4` (mỗi đỉnh bậc `3`) cần `4` màu: một bộ `3` màu sẽ có hai đỉnh cùng màu mà vẫn kề nhau. Nên giả thiết "bậc `≤ 3`" và "`4` loại hoa" khớp nhau chính xác.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `4^n` | `O(4^n · m)` | `O(n)` | oracle cho `n ≤ 8` |
| 2. Quay lui | `O(n + m)` (không lùi) | `O(n + m)` | tổng quát cho đồ thị bất kỳ |
| 3. Tham lam theo chỉ số | `O(n + m)` | `O(n + m)` | **đáp án chính** |
| 4. Tham lam theo BFS | `O(n + m)` | `O(n + m)` | chứng minh thứ tự bất kỳ |
| 5. Mặt nạ bit | `O(n + m)` | `O(n + m)` | hằng số nhỏ |

Do mỗi đỉnh có bậc `≤ 3` nên `m ≤ 3n / 2`, tức `O(n + m) = O(n)`. Đo thực tế với `n = 10^4`, `m = 15 000` (đồ thị `3`-đều):

In [17]:
def three_regular(n):
    """Đồ thị 3-đều trên n đỉnh (n chẵn): chu trình + dây cung i - (i + n/2)."""
    paths = [[i + 1, (i + 1) % n + 1] for i in range(n)]
    paths += [[i + 1, i + n // 2 + 1] for i in range(n // 2)]
    return paths


def bench(fn, *args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


N = 10_000
P = three_regular(N)
assert len(P) == 15_000 and max(len(a) for a in build_graph(N, P)) == 3
for name, fn in [("backtrack", garden_backtrack), ("greedy", garden_greedy), ("bfs", garden_bfs), ("bitmask", garden_bitmask)]:
    r = fn(N, P)
    assert is_valid(N, P, r)
    print(f"{name:10s}: {bench(fn, N, P) * 1000:7.1f} ms  hợp lệ")

backtrack :     7.8 ms  hợp lệ
greedy    :     4.4 ms  hợp lệ


bfs       :     7.7 ms  hợp lệ


bitmask   :     4.4 ms  hợp lệ


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ cho lỗi hay gặp, (D) vét cạn mọi đồ thị nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

Tất cả đều dùng **hàm hợp lệ** `is_valid`, không bao giờ so sánh với mảng mẫu.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "greedy": garden_greedy,
    "backtrack": garden_backtrack,
    "bfs": garden_bfs,
    "bitmask": garden_bitmask,
    "Solution": lambda n, p: Solution().gardenNoAdj(n, p),
    "SolutionGreedy": lambda n, p: SolutionGreedy().gardenNoAdj(n, p),
    "SolutionBacktrack": lambda n, p: SolutionBacktrack().gardenNoAdj(n, p),
    "SolutionBFS": lambda n, p: SolutionBFS().gardenNoAdj(n, p),
    "SolutionBitmask": lambda n, p: SolutionBitmask().gardenNoAdj(n, p),
}
SMALL_IMPLS = {"brute": garden_brute, "SolutionBrute": lambda n, p: SolutionBrute().gardenNoAdj(n, p)}


def check(n, paths, small=False):
    impls = dict(IMPLS)
    if small:
        impls.update(SMALL_IMPLS)
    for name, fn in impls.items():
        r = fn(n, paths)
        assert is_valid(n, paths, r), (name, n, paths, r)
    return True


for n, paths in examples:
    check(n, paths, small=True)
print("ví dụ đề: mọi cài đặt (kể cả brute) cho đáp án hợp lệ;", len(IMPLS) + len(SMALL_IMPLS), "cài đặt")

# Cùng ví dụ, so sánh bằng mảng mẫu SẼ SAI: có nhiều đáp án đúng
assert is_valid(3, [[1, 2], [2, 3], [3, 1]], [1, 2, 4]) and [1, 2, 4] != [1, 2, 3]

ví dụ đề: mọi cài đặt (kể cả brute) cho đáp án hợp lệ; 11 cài đặt


### B. Edge case

- `n = 1`, không có đường: mọi loại hoa đều hợp lệ (tham lam cho `[1]`).
- Không có đường đi nào: mọi vườn tự do (tham lam cho toàn `1`).
- Hai vườn nối nhau; đồ thị đường thẳng; chu trình lẻ (`C3`, `C5`); sao `K_{1,3}` (đỉnh giữa bậc `3`).
- `K4`: bắt buộc dùng đủ `4` loại hoa.
- Hai thành phần liên thông rời nhau; vườn cô lập xen kẽ.
- Hai bản `K4` ghép chung: mỗi đỉnh bậc `3`, đủ chặt.

In [19]:
edge_cases = [
    (1, []),
    (5, []),
    (2, [[1, 2]]),
    (6, [[1, 2], [2, 3], [3, 4], [4, 5], [5, 6]]),                                # đường thẳng
    (3, [[1, 2], [2, 3], [3, 1]]),                                                # C3
    (5, [[1, 2], [2, 3], [3, 4], [4, 5], [5, 1]]),                                # C5
    (4, [[1, 2], [1, 3], [1, 4]]),                                                # sao K_{1,3}
    (4, [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]),                        # K4
    (7, [[1, 2], [3, 4], [5, 6]]),                                                # thành phần rời + đỉnh cô lập
    (8, [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4], [5, 6], [5, 7], [5, 8], [6, 7], [6, 8], [7, 8]]),   # hai K4
    (4, [[4, 3], [3, 2], [2, 1]]),                                                # cạnh viết ngược chiều
    (5, [[1, 5], [5, 2], [2, 4], [4, 3]]),                                        # nhãn không theo thứ tự đường đi
]
for n, paths in edge_cases:
    check(n, paths, small=(n <= 8))
print(len(edge_cases), "edge case: mọi cài đặt cho đáp án hợp lệ")
print("K4 ->", Solution().gardenNoAdj(*edge_cases[7]), "| n=1 ->", Solution().gardenNoAdj(1, []), "| n=5 rỗng ->", Solution().gardenNoAdj(5, []))
assert sorted(Solution().gardenNoAdj(*edge_cases[7])) == [1, 2, 3, 4]              # K4 buộc dùng đủ 4 loại

12 edge case: mọi cài đặt cho đáp án hợp lệ
K4 -> [1, 2, 3, 4] | n=1 -> [1] | n=5 rỗng -> [1, 1, 1, 1, 1]


### C. Phản ví dụ: cài đặt sai phải bị `is_valid` bắt

In [20]:
def bug_one_direction(n, paths):                        # chỉ thêm cạnh một chiều
    g = defaultdict(list)
    for x, y in paths:
        g[x - 1].append(y - 1)
    ans = [0] * n
    for x in range(n):
        used = {ans[y] for y in g[x]}
        ans[x] = next(c for c in range(1, 5) if c not in used)
    return ans


def bug_three_colors(n, paths):                         # chỉ dùng 3 màu
    g = build_graph(n, paths)
    ans = [0] * n
    for x in range(n):
        used = {ans[y] for y in g[x]}
        ans[x] = next((c for c in range(1, 4) if c not in used), 0)     # K4: đỉnh cuối không có màu
    return ans


def bug_mod_four(n, paths):                             # tô theo chỉ số, không nhìn hàng xóm
    return [i % 4 + 1 for i in range(n)]


def bug_all_ones(n, paths):                             # coi như không có ràng buộc
    return [1] * n


def bug_index_shift(n, paths):                          # dùng nhãn 1-based làm chỉ số 0-based
    g = [[] for _ in range(n + 1)]
    for x, y in paths:
        g[x].append(y)
        g[y].append(x)
    ans = [0] * (n + 1)
    for x in range(n):                                  # tô các đỉnh 0..n-1, bỏ sót đỉnh n; đỉnh 0 không tồn tại
        used = {ans[y] for y in g[x]}
        ans[x] = next(c for c in range(1, 5) if c not in used)
    return ans[1:]                                      # lệch: ans[0] là đỉnh "0" ảo, vườn n không được tô (0)


def bug_neighbors_only_prev_one(n, paths):              # chỉ tránh màu của MỘT hàng xóm
    g = build_graph(n, paths)
    ans = [0] * n
    for x in range(n):
        first = g[x][0] if g[x] else None
        ans[x] = 1 if first is None or ans[first] != 1 else 2
    return ans


BUGS = [bug_one_direction, bug_three_colors, bug_mod_four, bug_all_ones, bug_index_shift, bug_neighbors_only_prev_one]


def all_small_graphs(n):
    """Mọi đồ thị trên n đỉnh có bậc tối đa 3 (n <= 6)."""
    edges = [(i, j) for i in range(1, n + 1) for j in range(i + 1, n + 1)]
    for mask in range(1 << len(edges)):
        deg = [0] * (n + 1)
        es = []
        ok = True
        for b, (i, j) in enumerate(edges):
            if mask >> b & 1:
                deg[i] += 1
                deg[j] += 1
                if deg[i] > 3 or deg[j] > 3:
                    ok = False
                    break
                es.append([i, j])
        if ok:
            yield es


pool = [(n, es) for n in range(1, 6) for es in all_small_graphs(n)]
for bug in BUGS:
    found = None
    for n, es in pool:
        try:
            r = bug(n, es)
        except IndexError:
            found = (n, es, "IndexError")
            break
        if not is_valid(n, es, r):
            found = (n, es, r)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:30s} sai trên n = {found[0]}, paths = {found[1]}: ra {found[2]}")

bug_one_direction              sai trên n = 2, paths = [[1, 2]]: ra [1, 1]
bug_three_colors               sai trên n = 4, paths = [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]: ra [1, 2, 3, 0]
bug_mod_four                   sai trên n = 5, paths = [[1, 5]]: ra [1, 2, 3, 4, 1]
bug_all_ones                   sai trên n = 2, paths = [[1, 2]]: ra [1, 1]
bug_index_shift                sai trên n = 1, paths = []: ra [0]
bug_neighbors_only_prev_one    sai trên n = 3, paths = [[1, 2], [1, 3], [2, 3]]: ra [1, 2, 2]


### D. Vét cạn mọi đồ thị nhỏ

Mọi đồ thị trên `n = 1..6` đỉnh có bậc tối đa `3` (với `n = 6`: `2^15 = 32 768` tập cạnh, lọc theo bậc). Với `n ≤ 5` ta so cả brute `4^n`; mọi cài đặt phải cho đáp án hợp lệ. Ngoài ra kiểm: tham lam theo chỉ số = quay lui = mặt nạ bit (đúng cùng một đáp án), và (với `n ≤ 5`) = **đáp án nhỏ nhất theo từ điển** của brute.

In [21]:
t0 = time.perf_counter()
cnt = 0
for n in range(1, 7):
    for es in all_small_graphs(n):
        gr = garden_greedy(n, es)
        bt, backs = garden_backtrack(n, es, count_back=True)
        bm = garden_bitmask(n, es)
        assert is_valid(n, es, gr) and gr == bt == bm and backs == 0, (n, es)
        for name in ("Solution", "SolutionGreedy", "SolutionBacktrack", "SolutionBitmask"):
            assert IMPLS[name](n, es) == gr
        assert is_valid(n, es, garden_bfs(n, es)) and is_valid(n, es, IMPLS["SolutionBFS"](n, es))
        if n <= 5:
            assert garden_brute(n, es) == gr, (n, es)          # tham lam = nhỏ nhất theo từ điển
        cnt += 1
print(cnt, "đồ thị (n = 1..6, bậc <= 3):", "tham lam = quay lui = mặt nạ bit; nhỏ nhất theo từ điển (n<=5); quay lui không lùi;", round(time.perf_counter() - t0, 1), "s")

12911 đồ thị (n = 1..6, bậc <= 3): tham lam = quay lui = mặt nạ bit; nhỏ nhất theo từ điển (n<=5); quay lui không lùi; 0.7 s


### E. Tính chất / ngẫu nhiên (seed cố định)

Sinh đồ thị ngẫu nhiên có bậc `≤ 3` (không cạnh trùng). Kiểm: (1) mọi cài đặt cho đáp án hợp lệ; (2) tô theo **hoán vị đỉnh ngẫu nhiên** luôn hợp lệ; (3) đổi nhãn đỉnh (đánh số lại ngẫu nhiên) vẫn hợp lệ; (4) đảo thứ tự các đường và hướng từng đường không làm hỏng tính hợp lệ; (5) với `n ≤ 7`, brute `4^n` cũng cho đáp án hợp lệ và tham lam trùng nhỏ nhất theo từ điển.

In [22]:
def random_graph(rng, n, density):
    """Đồ thị ngẫu nhiên, bậc <= 3, không cạnh trùng; density in [0, 1] điều chỉnh số lần thử thêm cạnh."""
    deg = [0] * (n + 1)
    seen = set()
    paths = []
    tries = int(density * 3 * n)
    for _ in range(tries):
        if n < 2:
            break
        x, y = rng.sample(range(1, n + 1), 2)
        if deg[x] >= 3 or deg[y] >= 3 or (min(x, y), max(x, y)) in seen:
            continue
        seen.add((min(x, y), max(x, y)))
        deg[x] += 1
        deg[y] += 1
        paths.append([x, y] if rng.random() < 0.5 else [y, x])
    return paths


rng = random.Random(1042)
t0 = time.perf_counter()
cnt = 0
for it in range(1500):
    n = rng.randint(1, 60)
    paths = random_graph(rng, n, rng.choice([0.2, 0.5, 1.0, 2.0]))
    assert max([0] + [len(a) for a in build_graph(n, paths)]) <= 3
    check(n, paths, small=False)                                                       # (1)
    g = build_graph(n, paths)
    order = list(range(n))
    rng.shuffle(order)
    assert is_valid(n, paths, color_in_order(g, order))                                # (2)
    perm = list(range(1, n + 1))
    rng.shuffle(perm)
    relabeled = [[perm[x - 1], perm[y - 1]] for x, y in paths]
    assert is_valid(n, relabeled, Solution().gardenNoAdj(n, relabeled))               # (3)
    shuffled = [p[::-1] if rng.random() < 0.5 else p for p in paths]
    rng.shuffle(shuffled)
    assert is_valid(n, shuffled, Solution().gardenNoAdj(n, shuffled))                 # (4)
    cnt += 1
for it in range(300):
    n = rng.randint(1, 7)
    paths = random_graph(rng, n, rng.choice([0.5, 1.0, 2.0]))
    assert is_valid(n, paths, garden_brute(n, paths))
    assert garden_brute(n, paths) == garden_greedy(n, paths)                          # (5)
print(cnt + 300, "đồ thị ngẫu nhiên: 5 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

1800 đồ thị ngẫu nhiên: 5 tính chất đúng; 0.9 s


### F. Giới hạn

`n = 10^4`: không có đường; đường thẳng; chu trình; đồ thị `3`-đều `15 000` cạnh; `2 500` bản `K4` rời nhau (mỗi bản bắt buộc đủ `4` màu); nhãn đảo ngược; đồ thị ngẫu nhiên đậm đặc. Mọi cài đặt (trừ brute) phải hợp lệ và chạy nhanh.

In [23]:
t0 = time.perf_counter()
N = 10_000
big_cases = {
    "không đường": (N, []),
    "đường thẳng": (N, [[i, i + 1] for i in range(1, N)]),
    "chu trình": (N, [[i, i % N + 1] for i in range(1, N + 1)]),
    "3-đều 15000 cạnh": (N, three_regular(N)),
    "2500 K4": (N, [[4 * b + i, 4 * b + j] for b in range(N // 4) for i in range(1, 5) for j in range(i + 1, 5)]),
    "3-đều, nhãn đảo": (N, [[N + 1 - x, N + 1 - y] for x, y in three_regular(N)]),
}
rg = random.Random(7)
big_cases["ngẫu nhiên đậm đặc"] = (N, random_graph(rg, N, 2.0))
for name, (n, paths) in big_cases.items():
    assert max([0] + [len(a) for a in build_graph(n, paths)]) <= 3
    check(n, paths)
    print(f"  {name:20s}: n = {n}, m = {len(paths):5d}: hợp lệ")
k4 = Solution().gardenNoAdj(*big_cases["2500 K4"])
assert all(sorted(k4[4 * b: 4 * b + 4]) == [1, 2, 3, 4] for b in range(N // 4))
print("n = 10^4:", round(time.perf_counter() - t0, 1), "s")

  không đường         : n = 10000, m =     0: hợp lệ


  đường thẳng         : n = 10000, m =  9999: hợp lệ
  chu trình           : n = 10000, m = 10000: hợp lệ


  3-đều 15000 cạnh    : n = 10000, m = 15000: hợp lệ


  2500 K4             : n = 10000, m = 15000: hợp lệ
  3-đều, nhãn đảo     : n = 10000, m = 15000: hợp lệ


  ngẫu nhiên đậm đặc  : n = 10000, m = 14064: hợp lệ


n = 10^4: 0.8 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ thêm cạnh một chiều

In [24]:
n, paths = 2, [[1, 2]]
print("bug:", bug_one_direction(n, paths), "| đúng:", Solution().gardenNoAdj(n, paths))
assert not is_valid(n, paths, bug_one_direction(n, paths)) and is_valid(n, paths, Solution().gardenNoAdj(n, paths))

bug: [1, 1] | đúng: [1, 2]


**Vì sao sai:** con đường là **hai chiều**. Nếu chỉ `g[x].append(y)` thì khi tô `y` (đến sau `x`) ta không thấy `x` là hàng xóm và có thể trồng cùng loại với `x`. **Đúng:** luôn thêm cả `g[x].append(y)` và `g[y].append(x)`.

### Lỗi 2: chỉ dùng 3 loại hoa

In [25]:
n, paths = edge_cases[7]                                 # K4
print("bug:", bug_three_colors(n, paths), "| đúng:", Solution().gardenNoAdj(n, paths))
assert not is_valid(n, paths, bug_three_colors(n, paths))

bug: [1, 2, 3, 0] | đúng: [1, 2, 3, 4]


**Vì sao sai:** nghĩ "bậc `3` nên cần `3` màu". Đỉnh có `3` hàng xóm đã tô **khác màu nhau** cần màu thứ `4`; `K4` buộc phải dùng đủ `4` loại. Công thức đúng: cần `Δ + 1` màu.

### Lỗi 3: lệch chỉ số `1`-based và `0`-based

In [26]:
n, paths = 3, [[1, 2], [2, 3], [3, 1]]
print("bug:", bug_index_shift(n, paths), "| đúng:", Solution().gardenNoAdj(n, paths))
assert not is_valid(n, paths, bug_index_shift(n, paths))

bug: [1, 2, 0] | đúng: [1, 2, 3]


**Vì sao sai:** nhãn vườn chạy `1..n` còn mảng `ans` chạy `0..n-1`. Quên trừ `1` làm lệch: có vườn không bao giờ được tô (còn `0` — không phải loại hoa hợp lệ) và đỉnh "ảo" `0` bị tô thừa. **Đúng:** đổi ngay `x - 1`, `y - 1` khi dựng đồ thị.

### Lỗi 4: không nhìn hàng xóm (tô theo `i % 4`)

In [27]:
n, paths = 5, [[1, 5]]
print("bug:", bug_mod_four(n, paths), "| đúng:", Solution().gardenNoAdj(n, paths))
assert not is_valid(n, paths, bug_mod_four(n, paths))

bug: [1, 2, 3, 4, 1] | đúng: [1, 1, 1, 1, 2]


**Vì sao sai:** vườn `1` (`i = 0`) và vườn `5` (`i = 4`) cùng nhận loại `1` vì `0 % 4 = 4 % 4`, nhưng chúng lại nối nhau. Màu phải phụ thuộc **các hàng xóm**, không phụ thuộc chỉ số.

### Lỗi 5: so sánh đáp án với mảng mẫu của đề

In [28]:
n, paths = examples[0]
mine = [1, 2, 4]                                          # hợp lệ nhưng khác mảng mẫu [1, 2, 3]
print("so sánh bằng:", mine == sample_answers[0], "| is_valid:", is_valid(n, paths, mine))
assert mine != sample_answers[0] and is_valid(n, paths, mine)

so sánh bằng: False | is_valid: True


**Vì sao sai:** đề yêu cầu "**any** such a choice". Đáp án của bạn có thể khác mảng mẫu mà vẫn đúng. Khi tự test, luôn dùng hàm kiểm tra hợp lệ (như `is_valid`), không `assert ans == expected`.

### Lỗi 6 (bonus): chỉ tránh màu của một hàng xóm

In [29]:
bad = next(((n, es) for n, es in pool if not is_valid(n, es, bug_neighbors_only_prev_one(n, es))), None)
print("phản ví dụ:", bad, "-> bug:", bug_neighbors_only_prev_one(*bad), "| đúng:", Solution().gardenNoAdj(*bad))
assert bad is not None and is_valid(*bad, Solution().gardenNoAdj(*bad))

phản ví dụ: (3, [[1, 2], [1, 3], [2, 3]]) -> bug: [1, 2, 2] | đúng: [1, 2, 3]


**Vì sao sai:** phải tránh màu của **mọi** hàng xóm đã tô, không chỉ một. Dùng tập `used` tổng hợp tất cả.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "gán nhãn cho đỉnh sao cho hai đỉnh kề khác nhãn; số nhãn ≥ bậc lớn nhất + 1" → **tô màu tham lam**, không cần quay lui. Đề nói "đảm bảo tồn tại đáp án" và "trả về bất kỳ" là dấu hiệu bài có nhiều nghiệm.

**Mẹo:**

1. Số màu `≥ Δ + 1` ⇒ tham lam luôn thành công, theo bất kỳ thứ tự đỉnh nào.
2. Dựng đồ thị `0`-based ngay từ đầu; thêm cạnh **hai chiều**.
3. Dùng `set` (hoặc mặt nạ bit) cho màu bị cấm; màu `0` = chưa tô nên vô hại.
4. Bài nhiều đáp án: test bằng **hàm hợp lệ**, thêm brute `4^n` trên đồ thị nhỏ làm đối chứng.
5. Muốn đáp án nhỏ nhất theo từ điển: tham lam theo chỉ số tăng dần (mục 10e).
6. Bất kể đồ thị có bao nhiêu thành phần liên thông, vòng `for x in range(n)` phủ đủ; không cần DFS riêng cho từng thành phần.

**Họ bài cùng pattern:** Is Graph Bipartite? (785), Possible Bipartition (886), Course Schedule (207), Number of Provinces (547), Graph Valid Tree (261), Palindrome Partitioning (131).

## 15. Biến thể

1. **`Δ + 1` màu cho đồ thị bất kỳ:** tô tham lam với `k = Δ + 1` màu luôn hợp lệ (tổng quát hoá bài này).
2. **Đáp án nhỏ nhất theo từ điển:** chính là tô tham lam theo chỉ số (kiểm bằng vét cạn).
3. **Đếm số cách tô** bằng `k` màu cho đồ thị nhỏ (đa thức sắc độ): đường thẳng `n` đỉnh có `k·(k−1)^(n−1)` cách; `K4` với `4` màu có `24` cách.
4. **Kiểm tra `2`-tô màu được** (đồ thị hai phía) bằng BFS: chu trình lẻ thì không.

Cài đặt và đối chiếu bằng vét cạn:

In [30]:
def greedy_k(n, paths, k):
    """Tô tham lam với k màu 1..k; trả về None nếu có đỉnh không còn màu trống."""
    g = build_graph(n, paths)
    ans = [0] * n
    for x in range(n):
        used = {ans[y] for y in g[x]}
        c = next((c for c in range(1, k + 1) if c not in used), None)
        if c is None:
            return None
        ans[x] = c
    return ans


def valid_k(n, paths, ans, k):
    return ans is not None and len(ans) == n and all(1 <= c <= k for c in ans) and all(ans[x - 1] != ans[y - 1] for x, y in paths)


def count_colorings(n, paths, k):
    g = build_graph(n, paths)
    ans = [0] * n

    def go(i):
        if i == n:
            return 1
        total = 0
        for c in range(1, k + 1):
            if all(ans[y] != c for y in g[i] if y < i):
                ans[i] = c
                total += go(i + 1)
                ans[i] = 0
        return total

    return go(0)


def is_bipartite(n, paths):
    g = build_graph(n, paths)
    side = [-1] * n
    for s in range(n):
        if side[s] != -1:
            continue
        side[s] = 0
        dq = deque([s])
        while dq:
            u = dq.popleft()
            for v in g[u]:
                if side[v] == -1:
                    side[v] = side[u] ^ 1
                    dq.append(v)
                elif side[v] == side[u]:
                    return False
    return True


rng = random.Random(15)
# (1) Δ+1 màu cho đồ thị bất kỳ (bậc có thể lớn)
for _ in range(300):
    n = rng.randint(1, 25)
    es = [[a, b] for a in range(1, n + 1) for b in range(a + 1, n + 1) if rng.random() < 0.25]
    delta = max([0] + [len(a) for a in build_graph(n, es)])
    assert valid_k(n, es, greedy_k(n, es, delta + 1), delta + 1)
# (3) đếm số cách
for n in range(1, 8):
    path = [[i, i + 1] for i in range(1, n)]
    for k in (2, 3, 4):
        assert count_colorings(n, path, k) == k * (k - 1) ** (n - 1)
assert count_colorings(4, edge_cases[7][1], 4) == 24 and count_colorings(4, edge_cases[7][1], 3) == 0
assert count_colorings(3, [[1, 2], [2, 3], [3, 1]], 4) == 24
# (4) hai phía  <=>  tồn tại 2-tô màu (vét cạn)
for n in range(1, 6):
    for es in all_small_graphs(n):
        brute2 = any(all(a[x - 1] != a[y - 1] for x, y in es) for a in itertools.product((1, 2), repeat=n))
        assert is_bipartite(n, es) == brute2
print("Δ+1 màu tham lam đúng; số cách tô khớp công thức; kiểm tra hai phía khớp vét cạn")

Δ+1 màu tham lam đúng; số cách tô khớp công thức; kiểm tra hai phía khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử `4^n` bộ | `O(4^n · m)` | `O(n)` | oracle `n ≤ 8` |
| 2. Quay lui | thử màu, lùi khi kẹt | `O(n + m)` ở bài này | `O(n + m)` | đồ thị tổng quát (không bảo đảm đủ màu) |
| 3. Tham lam theo chỉ số | màu nhỏ nhất không bị cấm | `O(n + m)` | `O(n + m)` | **đáp án chính**; nhỏ nhất theo từ điển |
| 4. Tham lam theo BFS | thứ tự thăm bất kỳ | `O(n + m)` | `O(n + m)` | chứng tỏ không phụ thuộc thứ tự |
| 5. Mặt nạ bit | `free & -free` | `O(n + m)` | `O(n + m)` | hằng số nhỏ |

Mọi cách (trừ vét cạn khi `n > 8`) cho đáp án hợp lệ trên toàn bộ đồ thị bậc `≤ 3` có `n ≤ 6` và các đồ thị ngẫu nhiên / lớn.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell. Nhớ kiểm tra bằng `is_valid`, không so sánh với mảng mẫu.

### Bài 1 (đoán output)

Đoán trước rồi chạy (đáp án tham lam theo chỉ số):

In [31]:
print(Solution().gardenNoAdj(4, [[1, 2], [3, 4]]))
print(Solution().gardenNoAdj(4, [[1, 2], [1, 3], [1, 4]]))
print(Solution().gardenNoAdj(3, [[1, 2], [2, 3], [3, 1]]))

[1, 2, 1, 2]
[1, 2, 2, 2]
[1, 2, 3]


### Bài 2 (tính tay)

`n = 5`, `paths = [[1,2],[2,3],[3,4],[4,5],[5,1]]` (chu trình độ dài `5`). Chạy tham lam theo chỉ số bằng tay: viết `used` và màu của từng vườn. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tô tham lam:

In [32]:
def garden_todo(n, paths):
    g = [[] for _ in range(n)]
    for x, y in paths:
        pass            # TODO 1: thêm cạnh hai chiều (nhớ đổi sang 0-based)
    ans = [0] * n
    for x in range(n):
        pass            # TODO 2: used = màu của hàng xóm; gán màu nhỏ nhất trong 1..4 không thuộc used
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(is_valid(n, p, garden_todo(n, p)) for n, p in examples + edge_cases)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` tô theo **thứ tự BFS** (hoặc thứ tự bất kỳ):

In [33]:
class MySolution:
    def gardenNoAdj(self, n, paths):
        pass            # TODO


# Khi làm xong:
# assert all(is_valid(n, p, MySolution().gardenNoAdj(n, p)) for n, p in examples + edge_cases)

### Bài 5 (mini challenge): đếm số cách trồng hoa

Viết `count_ways(n, paths, k)` trả về số cách trồng hoa hợp lệ khi có `k` loại hoa, cho `n ≤ 8`. Kiểm tra: tam giác với `4` loại → `24`; đường thẳng `n` đỉnh → `k·(k−1)^(n−1)`.

In [34]:
def count_ways(n, paths, k):
    pass            # TODO (quay lui hoặc itertools.product)


# Khi làm xong:
# assert count_ways(3, [[1, 2], [2, 3], [3, 1]], 4) == 24
# assert count_ways(4, [[1, 2], [2, 3], [3, 4]], 3) == 3 * 2 ** 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tô theo chỉ số tăng dần, mỗi vườn lấy màu nhỏ nhất chưa dùng bởi hàng xóm đã tô.
- Mức 2: ví dụ sao: vườn `1` được `1`, các vườn `2, 3, 4` chỉ kề vườn `1` nên đều lấy màu nhỏ nhất khác `1`.
- Mức 3: `[1, 2, 1, 2]`, `[1, 2, 2, 2]`, `[1, 2, 3]`.

**Bài 2**

- Mức 1: vườn `1`: không hàng xóm nào đã tô ⇒ `1`.
- Mức 2: vườn `2` (hàng xóm `1`) ⇒ `2`; vườn `3` (hàng xóm `2`) ⇒ `1`; vườn `4` (hàng xóm `3`) ⇒ `2`.
- Mức 3: vườn `5` có hàng xóm `4` (màu `2`) và `1` (màu `1`) nên `used = {1, 2}` ⇒ màu `3`; kết quả `[1, 2, 1, 2, 3]`.

**Bài 3**

- Mức 1: `g[x-1].append(y-1); g[y-1].append(x-1)`.
- Mức 2: `used = {ans[y] for y in g[x]}`.
- Mức 3: `ans[x] = next(c for c in range(1, 5) if c not in used)`.

**Bài 4**

- Mức 1: BFS từ mỗi đỉnh chưa tô; khi lấy đỉnh `u` ra khỏi hàng đợi thì tô nó.
- Mức 2: `used` = màu các hàng xóm đã tô (khác `0`).
- Mức 3: đẩy các hàng xóm chưa tô vào hàng đợi (dùng tập `seen` để không đẩy trùng).

**Bài 5**

- Mức 1: tô lần lượt các đỉnh `0..n-1`, thử `k` màu.
- Mức 2: chỉ kiểm tra hàng xóm có chỉ số nhỏ hơn (đã tô).
- Mức 3: đệ quy `go(i)`: cộng dồn `go(i + 1)` cho mỗi màu hợp lệ; `go(n) = 1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
r1 = Solution().gardenNoAdj(4, [[1, 2], [3, 4]])
r2 = Solution().gardenNoAdj(4, [[1, 2], [1, 3], [1, 4]])
r3 = Solution().gardenNoAdj(3, [[1, 2], [2, 3], [3, 1]])
print(r1, r2, r3)
assert r1 == [1, 2, 1, 2] and r2 == [1, 2, 2, 2] and r3 == [1, 2, 3]
assert is_valid(4, [[1, 2], [3, 4]], r1) and is_valid(4, [[1, 2], [1, 3], [1, 4]], r2) and is_valid(3, [[1, 2], [2, 3], [3, 1]], r3)
print("Bài 1 OK")

[1, 2, 1, 2] [1, 2, 2, 2] [1, 2, 3]
Bài 1 OK


### Bài 2

In [36]:
paths5 = [[1, 2], [2, 3], [3, 4], [4, 5], [5, 1]]
r = Solution().gardenNoAdj(5, paths5)
print(r)
assert r == [1, 2, 1, 2, 3] and is_valid(5, paths5, r)
assert r == garden_brute(5, paths5)                       # nhỏ nhất theo từ điển
print("Bài 2 OK: [1, 2, 1, 2, 3]")

[1, 2, 1, 2, 3]
Bài 2 OK: [1, 2, 1, 2, 3]


### Bài 3

In [37]:
def garden_answer(n, paths):
    g = [[] for _ in range(n)]
    for x, y in paths:
        g[x - 1].append(y - 1)
        g[y - 1].append(x - 1)
    ans = [0] * n
    for x in range(n):
        used = {ans[y] for y in g[x]}
        ans[x] = next(c for c in range(1, 5) if c not in used)
    return ans


assert all(is_valid(n, p, garden_answer(n, p)) for n, p in examples + edge_cases)
for n, es in pool:
    assert garden_answer(n, es) == Solution().gardenNoAdj(n, es)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def gardenNoAdj(self, n, paths):
        g = [[] for _ in range(n)]
        for x, y in paths:
            g[x - 1].append(y - 1)
            g[y - 1].append(x - 1)
        ans = [0] * n
        for s in range(n):
            if ans[s]:
                continue
            q = deque([s])
            seen = {s}
            while q:
                u = q.popleft()
                used = {ans[v] for v in g[u]}
                ans[u] = next(c for c in (1, 2, 3, 4) if c not in used)
                for v in g[u]:
                    if v not in seen and not ans[v]:
                        seen.add(v)
                        q.append(v)
        return ans


assert all(is_valid(n, p, MySolutionAnswer().gardenNoAdj(n, p)) for n, p in examples + edge_cases)
for n, es in pool:
    assert is_valid(n, es, MySolutionAnswer().gardenNoAdj(n, es))
assert is_valid(N, three_regular(N), MySolutionAnswer().gardenNoAdj(N, three_regular(N)))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def count_ways(n, paths, k):
    return count_colorings(n, paths, k)


assert count_ways(3, [[1, 2], [2, 3], [3, 1]], 4) == 24
assert count_ways(4, [[1, 2], [2, 3], [3, 4]], 3) == 3 * 2 ** 3
brute_count = sum(1 for a in itertools.product(range(1, 5), repeat=4) if all(a[x - 1] != a[y - 1] for x, y in edge_cases[7][1]))
assert count_ways(*edge_cases[7], 4) == brute_count == 24
for n, es in pool:
    if n <= 4:
        for k in (2, 3):
            bc = sum(1 for a in itertools.product(range(1, k + 1), repeat=n) if all(a[x - 1] != a[y - 1] for x, y in es))
            assert count_ways(n, es, k) == bc
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tô màu đồ thị bậc ≤ Δ với Δ + 1 màu, bài nhiều đáp án):**

```python
class Solution:
    def gardenNoAdj(self, n: int, paths: List[List[int]]) -> List[int]:
        g = defaultdict(list)
        for x, y in paths:
            x, y = x - 1, y - 1
            g[x].append(y)
            g[y].append(x)
        ans = [0] * n
        for x in range(n):
            used = {ans[y] for y in g[x]}
            for c in range(1, 5):
                if c not in used:
                    ans[x] = c
                    break
        return ans
```

**Checklist trước khi submit:**

- [ ] Thêm cạnh **hai chiều** và đổi sang `0`-based (`x - 1`, `y - 1`)?
- [ ] `used` lấy từ **mọi** hàng xóm (màu `0` = chưa tô là vô hại)?
- [ ] Dùng đủ `4` màu `1..4` (bậc `≤ 3` nên luôn còn màu trống)?
- [ ] Trả về mảng độ dài `n`, `ans[i]` ứng với vườn `i + 1`, không còn giá trị `0`?
- [ ] Test bằng hàm hợp lệ (`is_valid`), không so với mảng mẫu của đề; thử `K4`, `n = 1`, không đường?
- [ ] Đúng tên `class Solution`, `gardenNoAdj(self, n, paths)`, không in debug.